<a href="https://colab.research.google.com/github/osvaldomaguey/RappiPlus-Perfomance-Analysis/blob/main/S12_Estudiante_Proyecto_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python)

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [ ]:
# importar librerías
import pandas as pd


In [ ]:
# cargar archivos
orders = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv") # tu código aquí
catalog = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv") # tu código aquí
marketing = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv") # tu código aquí

In [ ]:
# explorar datasets
# tu código aquí
print(orders.info())
print()
print(orders.head(5))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB
None

  id_pedido id_usuario fecha_hora_pedido       pais dispositivo  \
0   order_0  user_6993        2025-05-22  Argentina     desktop   
1   order_1  

In [ ]:
print(catalog.info())
print()
print(catalog.head(5))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes
None

        nombre_producto categoria_producto  costo_unitario  \
0    Laptop-Gaming-16GB        Electrónica          280.68   
1       Phone-Pro-128GB        Electrónica           10.12   
2  Tablet-Standard-64GB        Electrónica           25.21   
3        Blender-XL-Red              Hogar          176.64   
4      Vacuum-Pro-Black              Hogar           16.60   

                 proveedor  
0   Fuller, Pena and Myers  
1                 King Ltd  
2               Bowers LLC  
3                Long-Reid  
4  Rivera, Carr and Finle

In [ ]:
print(marketing.info())
print()
print(marketing.head(5))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB
None

        fecha      pais            id_campaña        canal    gasto
0  2025-01-01    Mexico        organic_Mexico      organic  2446.25
1  2025-01-01    Mexico    paid_search_Mexico  paid_search  2704.34
2  2025-01-01    Mexico         social_Mexico       social  2045.01
3  2025-01-01  Colombia      organic_Colombia      organic  2597.21
4  2025-01-01  Colombia  paid_search_Colombia  paid_search  1771.40


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas

---

In [ ]:
# tu código aquí
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'], errors = 'coerce')
marketing['fecha'] = pd.to_datetime(marketing['fecha'], errors = 'coerce')
print(orders.info())
print("\n" + "_"*25 + "\n")
print(marketing.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25100 non-null  object        
 1   id_usuario          25100 non-null  object        
 2   fecha_hora_pedido   25100 non-null  datetime64[ns]
 3   pais                24800 non-null  object        
 4   dispositivo         25080 non-null  object        
 5   fuente_referencia   25070 non-null  object        
 6   nombre_producto     25070 non-null  object        
 7   categoria_producto  25020 non-null  object        
 8   cantidad            25050 non-null  float64       
 9   precio_unitario     25050 non-null  float64       
 10  monto_descuento     25050 non-null  float64       
 11  monto_total         25100 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.3+ MB
None

_________________________

<cl

In [ ]:
variables_numericas = ['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']
filas_con_nan = orders[orders[variables_numericas].isna().any(axis=1)]
print("\n=== FILAS COMPLETAS CON NaN ===")
filas_con_nan


=== FILAS COMPLETAS CON NaN ===


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
74,order_74,user_6172,2025-01-15,Argentina,desktop,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,595.85
75,order_75,user_6588,2025-06-19,Colombia,mobile,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,458.15
76,order_76,user_3193,2025-03-17,Argentina,mobile,paid_search,Laptop-Gaming-16GB,NaN,NaN,NaN,NaN,319.75
77,order_77,user_775,2025-06-24,Argentina,desktop,social,Vacuum-Pro-Black,NaN,NaN,NaN,NaN,227.55
78,order_78,user_2702,2025-03-19,Argentina,desktop,paid_search,Phone-Pro-128GB,NaN,NaN,NaN,NaN,432.39
79,order_79,user_6438,2025-03-22,Colombia,desktop,paid_search,Sneakers-Urban-42,NaN,NaN,NaN,NaN,527.15
80,order_80,user_6790,2025-01-03,Mexico,desktop,organic,Vacuum-Pro-Black,NaN,NaN,NaN,NaN,711.18
81,order_81,user_232,2025-05-16,Mexico,mobile,paid_search,Tablet-Standard-64GB,NaN,NaN,NaN,NaN,41.08
82,order_82,user_5665,2025-02-07,Colombia,mobile,paid_search,Sneakers-Urban-42,NaN,NaN,NaN,NaN,431.55
83,order_83,user_5225,2025-04-25,Argentina,mobile,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,279.35


In [ ]:
print('Orders variables numéricas')
print(orders[['cantidad','precio_unitario','monto_descuento','monto_total']].describe())
print("\n"+"-"*30+"\n")
print('Catalog variables numéricas')
print(catalog['costo_unitario'].describe())
print("\n"+"-"*30+"\n")
print('Marketing variables numéricas')
print(marketing['gasto'].describe())

Orders variables numéricas
           cantidad  precio_unitario  monto_descuento   monto_total
count  25050.000000     25050.000000     25050.000000  2.510000e+04
mean       7.092735       259.305549         4.500798  2.072680e+03
std      296.277003       138.726461         5.223010  9.894995e+04
min       -2.000000        20.030000         0.000000 -4.926500e+02
25%        1.000000       138.377500         0.000000  1.805075e+02
50%        2.000000       258.715000         0.000000  3.417500e+02
75%        2.000000       380.332500        10.000000  5.185800e+02
max    20000.000000       499.960000        15.000000  8.840200e+06

------------------------------

Catalog variables numéricas
count      7.000000
mean     102.252857
std      111.011563
min       10.120000
25%       16.905000
50%       25.210000
75%      182.975000
max      280.680000
Name: costo_unitario, dtype: float64

------------------------------

Marketing variables numéricas
count    1620.00000
mean     1772.74292


In [ ]:
col_numericas_orders = ['cantidad','monto_total']
for cols in col_numericas_orders:
    print(f"Value Counts en {cols}:")
    print(orders[cols].value_counts(dropna=False))
    print("\n"+"-"*30+"\n")
print("Value Counts para valores negativos en MONTO_TOTAL en orders")
print(orders[orders['monto_total'] < 0]['monto_total'].value_counts())

Value Counts en cantidad:
 2.0        12642
 1.0        12394
 NaN           50
 10000.0        6
 20000.0        4
-1.0            3
-2.0            1
Name: cantidad, dtype: int64

------------------------------

Value Counts en monto_total:
95.59     5
65.85     5
53.89     4
392.00    4
214.38    4
         ..
165.49    1
274.61    1
752.11    1
904.74    1
679.33    1
Name: monto_total, Length: 21536, dtype: int64

------------------------------

Value Counts para valores negativos en MONTO_TOTAL en orders
-492.65    1
-423.53    1
-192.62    1
-38.50     1
Name: monto_total, dtype: int64


In [ ]:
orders_clean = orders.copy()
orders_clean = orders_clean[(orders_clean['cantidad'] > 0)
    & (orders_clean['monto_total'] > 0)]

In [ ]:
print('Orders variables numéricas')
print(orders_clean[['cantidad','precio_unitario','monto_descuento','monto_total']].describe())
print("\n"+"-"*30+"\n")

Orders variables numéricas
           cantidad  precio_unitario  monto_descuento   monto_total
count  25046.000000     25046.000000     25046.000000  2.504600e+04
mean       7.094067       259.304400         4.500719  2.076331e+03
std      296.300643       138.715188         5.223232  9.905653e+04
min        1.000000        20.030000         0.000000  5.240000e+00
25%        1.000000       138.405000         0.000000  1.804025e+02
50%        2.000000       258.715000         0.000000  3.415600e+02
75%        2.000000       380.272500        10.000000  5.184375e+02
max    20000.000000       499.960000        15.000000  8.840200e+06

------------------------------



In [ ]:
col_categoricas_orders = ['pais','dispositivo','fuente_referencia','nombre_producto','categoria_producto']
for cols in col_categoricas_orders:
    print(f"Value counts para {cols}:")
    print(orders_clean[cols].value_counts(dropna=False))
    print("\n"+"-"*30+"\n")

Value counts para pais:
Colombia     7506
Mexico       7489
Argentina    7271
mexico        864
colombia      822
argentina     798
NaN           296
Name: pais, dtype: int64

------------------------------

Value counts para dispositivo:
desktop    12729
mobile     12297
NaN           20
Name: dispositivo, dtype: int64

------------------------------

Value counts para fuente_referencia:
social         8411
organic        8310
paid_search    8295
NaN              30
Name: fuente_referencia, dtype: int64

------------------------------

Value counts para nombre_producto:
Vacuum-Pro-Black        4193
Blender-XL-Red          4192
Jacket-Winter-M         4182
Sneakers-Urban-42       4141
Laptop-Gaming-16GB      2790
Tablet-Standard-64GB    2775
Phone-Pro-128GB         2743
NaN                       30
Name: nombre_producto, dtype: int64

------------------------------

Value counts para categoria_producto:
Hogar          8385
Moda           8323
Electronica    8308
NaN              30
Nam

In [ ]:
orders_clean = orders_clean.dropna(subset=['categoria_producto', 'nombre_producto', 'fuente_referencia'])
orders_clean['categoria_producto'] = orders_clean['categoria_producto'].replace('Electronica', 'Electrónica')
print("Categorías después del cambio:")
print(orders_clean['categoria_producto'].value_counts(dropna=False))

Categorías después del cambio:
Hogar          8385
Moda           8323
Electrónica    8308
Name: categoria_producto, dtype: int64


In [ ]:
orders_clean['pais'] = orders_clean['pais'].str.title()
print(orders_clean['pais'].value_counts(dropna=False))

Mexico       8339
Colombia     8321
Argentina    8060
NaN           296
Name: pais, dtype: int64


In [ ]:
col_categoricas_catalog = ['nombre_producto','categoria_producto','proveedor']
print(catalog[col_categoricas_catalog].describe())
print("\n"+"-"*30+"\n")
for cols in col_categoricas_catalog:
    print(f"Value Counts para {cols}:")
    print(catalog[cols].value_counts(dropna=False))
    print("\n"+"-"*30+"\n")

          nombre_producto categoria_producto   proveedor
count                   7                  7           7
unique                  7                  3           7
top     Sneakers-Urban-42        Electrónica  Bowers LLC
freq                    1                  3           1

------------------------------

Value Counts para nombre_producto:
Sneakers-Urban-42       1
Jacket-Winter-M         1
Phone-Pro-128GB         1
Laptop-Gaming-16GB      1
Vacuum-Pro-Black        1
Tablet-Standard-64GB    1
Blender-XL-Red          1
Name: nombre_producto, dtype: int64

------------------------------

Value Counts para categoria_producto:
Electrónica    3
Hogar          2
Moda           2
Name: categoria_producto, dtype: int64

------------------------------

Value Counts para proveedor:
Bowers LLC                 1
Mcmillan-Rhodes            1
Rivera, Carr and Finley    1
King Ltd                   1
Fuller, Pena and Myers     1
Long-Reid                  1
Greene-Smith               1
Nam

In [ ]:
col_categoricas_marketing = ['pais','canal']
print(marketing[col_categoricas_marketing].describe())
print("\n"+"-"*30+"\n")
for col in col_categoricas_marketing:
    print(f"Value Counts para {col}:")
    print(marketing[col].value_counts(dropna=False))
    print("\n"+"-"*30+"\n")

             pais        canal
count        1620         1519
unique          3            3
top     Argentina  paid_search
freq          540          507

------------------------------

Value Counts para pais:
Argentina    540
Colombia     540
Mexico       540
Name: pais, dtype: int64

------------------------------

Value Counts para canal:
paid_search    507
social         506
organic        506
NaN            101
Name: canal, dtype: int64

------------------------------



In [ ]:
print("Filas duplicadas:")
print(orders_clean.duplicated().sum())
duplicados_logicos = orders_clean.duplicated(subset=['id_usuario','id_pedido', 'fecha_hora_pedido', 'nombre_producto'])
print(f"Duplicados lógicos encontrados: {duplicados_logicos.sum()}")

Filas duplicadas:
100
Duplicados lógicos encontrados: 100


In [ ]:
orders_clean = orders_clean.drop_duplicates()
print("Filas completamente duplicadas:")
print(orders_clean.duplicated().sum())
duplicados_logicos = orders_clean.duplicated(subset=['id_usuario','id_pedido', 'fecha_hora_pedido', 'nombre_producto'])
print(f"Duplicados lógicos encontrados: {duplicados_logicos.sum()}")

Filas completamente duplicadas:
0
Duplicados lógicos encontrados: 0


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [ ]:
# exportar datasets
orders_clean.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

In [ ]:
categorias_orders = set(orders_clean['categoria_producto'].dropna())
categorias_catalog = set(catalog['categoria_producto'].dropna())

print("Categorías solo en orders:")
print(categorias_orders - categorias_catalog)
print("\nCategorías solo en catalog:")
print(categorias_catalog - categorias_orders)

Categorías solo en orders:
set()

Categorías solo en catalog:
set()


---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)?
- ¿Cuál es el costo total?
- ¿Cuánto se ha invertido en marketing?
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden?
- ¿Cuál es la cantidad promedio de productos por orden?
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal?

In [ ]:
# tu código aquí
ingreso_total = orders_clean['monto_total'].sum()
costo_descuentos = orders_clean['monto_descuento'].sum()
costo_marketing = marketing['gasto'].sum()
ordenes_con_costo = orders_clean.merge(catalog, on='nombre_producto', how='left')
costo_productos = (ordenes_con_costo['cantidad'] * ordenes_con_costo['costo_unitario']).sum()
costo_total = costo_descuentos + costo_marketing + costo_productos
utilidad = ingreso_total - costo_total
print(f"Ingreso Total: ${ingreso_total:,.2f}")
print("\n"+"-"*30+"\n")
print(f"Costo Total: ${costo_total:,.2f}")
print("\n"+"-"*30+"\n")
print(f"Inversion total en Marketing: ${costo_marketing:,.2f}")
print("\n"+"-"*30+"\n")
print(f"Utilidad: ${utilidad:,.2f}")

Ingreso Total: $51,954,718.94

------------------------------

Costo Total: $46,108,107.54

------------------------------

Inversion total en Marketing: $2,871,843.53

------------------------------

Utilidad: $5,846,611.40


In [ ]:
ticket_promedio = orders_clean['monto_total'].median()
print(f"Ticket Promedio: ${ticket_promedio:,.2f}")
print()
promedio_productos_orden = orders_clean['cantidad'].median()
print(f"Cantidad de productos promedio por órden: {promedio_productos_orden}")

Ticket Promedio: $341.61

Cantidad de productos promedio por órden: 2.0


In [ ]:
productos_vendidos = orders_clean['nombre_producto'].value_counts(dropna=False)
print("=== ANÁLISIS DE PRODUCTOS VENDIDOS ===")
print(productos_vendidos)
print("\n" + "-"*30)
print(f"Producto más Vendido: {productos_vendidos.index[0]}")

=== ANÁLISIS DE PRODUCTOS VENDIDOS ===
Blender-XL-Red          4176
Vacuum-Pro-Black        4170
Jacket-Winter-M         4166
Sneakers-Urban-42       4129
Laptop-Gaming-16GB      2778
Tablet-Standard-64GB    2764
Phone-Pro-128GB         2733
Name: nombre_producto, dtype: int64

------------------------------
Producto más Vendido: Blender-XL-Red


In [ ]:
#¿Cuánto se ha gastado en marketing por canal?
marketing.groupby('canal')['gasto'].sum()

canal
organic        913533.01
paid_search    863088.21
social         918043.21
Name: gasto, dtype: float64

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [ ]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT nombre_evento, COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY usuarios_unicos DESC;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios_unicos
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [ ]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
SELECT nombre_evento,
    COUNT(DISTINCT id_usuario),
    ROUND((COUNT(DISTINCT id_usuario) * 100.0) /
        (SELECT COUNT(DISTINCT id_usuario)
        FROM events WHERE nombre_evento = 'first_visit'), 2) AS porcentaje_conversion
FROM events
GROUP BY nombre_evento
ORDER BY
    CASE nombre_evento
        WHEN 'first_visit' THEN 1
        WHEN 'select_item' THEN 2
        WHEN 'add_to_cart' THEN 3
        WHEN 'begin_checkout' THEN 4
        WHEN 'add_payment_info' THEN 5
        WHEN 'purchase' THEN 6
    END;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,nombre_evento,count,porcentaje_conversion
0,first_visit,7796,100.00
1,select_item,7582,97.26
2,add_to_cart,7634,97.92
3,begin_checkout,7208,92.46
4,add_payment_info,6250,80.17
5,purchase,6240,80.04


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users`
- `user_activity`

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT
    MIN(fecha_registro) as primera_fecha_registro,
    MAX(fecha_registro) as ultima_fecha_registro
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(10)

,primera_fecha_registro,ultima_fecha_registro
0,2025-01-01,2025-05-31


In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(10)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free
3,user_3,2025-03-04,Mexico,desktop,free
4,user_4,2025-02-27,Argentina,desktop,free
5,user_5,2025-02-05,Colombia,desktop,free
6,user_6,2025-01-27,Mexico,mobile,free
7,user_7,2025-05-20,Colombia,mobile,free
8,user_8,2025-01-23,Mexico,mobile,free
9,user_9,2025-04-19,Colombia,desktop,free


In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT
    MIN(fecha_actividad) as primera_fecha_actividad,
    MAX(fecha_actividad) as ultima_fecha_actividad
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(10)


,primera_fecha_actividad,ultima_fecha_actividad
0,2025-01-08,2025-06-28


In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(10)


,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1
3,user_0,2025-02-26,28,0
4,user_1,2025-01-14,7,0
5,user_1,2025-01-21,14,0
6,user_1,2025-01-28,21,1
7,user_1,2025-02-04,28,0
8,user_2,2025-03-19,7,0
9,user_2,2025-03-26,14,1


In [ ]:
# Retención semanal (usuarios activos cada semana después de su registro)
# ======================

query_cohort_retention_final = '''
WITH cohorte AS
    (SELECT id_usuario,
        TO_CHAR(DATE_TRUNC('month', CAST(fecha_registro AS DATE)), 'MM-YYYY') AS cohorte,
        fecha_registro
    FROM users),
actividad_semanal AS
    (SELECT ua.id_usuario,
        ua.dias_despues_registro,
        ua.activo,
        c.cohorte
    FROM user_activity AS ua
    INNER JOIN cohorte AS c
    ON ua.id_usuario = c.id_usuario
    WHERE ua.activo = 1)

SELECT
    cohorte,
    COUNT(DISTINCT CASE WHEN dias_despues_registro = 7 THEN id_usuario END) AS retenido_w1,
    COUNT(DISTINCT CASE WHEN dias_despues_registro = 14 THEN id_usuario END) AS retenido_w2,
    COUNT(DISTINCT CASE WHEN dias_despues_registro = 21 THEN id_usuario END) AS retenido_w3

FROM actividad_semanal
GROUP BY cohorte
ORDER BY cohorte;

'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,retenido_w1,retenido_w2,retenido_w3
0,01-2025,697,668,656
1,02-2025,611,609,635
2,03-2025,677,705,690
3,04-2025,680,697,663
4,05-2025,695,676,706


In [ ]:
# Porcentaje de Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohorte_retencion AS
    (SELECT id_usuario,
        TO_CHAR(DATE_TRUNC('month', CAST(fecha_registro AS DATE)), 'MM-YYYY') AS cohorte_retencion,
        fecha_registro
    FROM users),
actividad_semanal AS
    (SELECT ua.id_usuario,
        ua.dias_despues_registro,
        ua.activo,
        cr.cohorte_retencion
    FROM user_activity AS ua
    INNER JOIN cohorte_retencion AS cr
    ON ua.id_usuario = cr.id_usuario
    WHERE ua.activo = 1),
usuarios_retenidos_por_semana AS
    (SELECT cohorte_retencion,
    COUNT(DISTINCT CASE WHEN dias_despues_registro = 7 THEN id_usuario END) AS retenido_w1,
    COUNT(DISTINCT CASE WHEN dias_despues_registro = 14 THEN id_usuario END) AS retenido_w2,
    COUNT(DISTINCT CASE WHEN dias_despues_registro = 21 THEN id_usuario END) AS retenido_w3
    FROM actividad_semanal
    GROUP BY cohorte_retencion),
total_usuarios AS
    (SELECT cohorte_retencion,
         COUNT(DISTINCT id_usuario) AS total_usuarios
    FROM cohorte_retencion
    GROUP BY cohorte_retencion)
SELECT
    ur.cohorte_retencion,
    100.00 AS semana_0,
    ROUND((retenido_w1 * 100.0) / tu.total_usuarios, 2) AS semana_1,
    ROUND((retenido_w2 * 100.0) / tu.total_usuarios, 2) AS semana_2,
    ROUND((retenido_w3 * 100.0) / tu.total_usuarios, 2) AS semana_3
FROM usuarios_retenidos_por_semana AS ur
INNER JOIN total_usuarios AS tu
ON ur.cohorte_retencion = tu.cohorte_retencion;
'''

# Ejecutar la consulta
cohorte_porcentaje_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_porcentaje_final

,cohorte_retencion,semana_0,semana_1,semana_2,semana_3
0,01-2025,100.0,42.84,41.06,40.32
1,02-2025,100.0,42.31,42.17,43.98
2,03-2025,100.0,41.38,43.09,42.18
3,04-2025,100.0,42.34,43.40,41.28
4,05-2025,100.0,41.20,40.07,41.85


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado**
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** ...
   - **H₁ (Hipótesis alternativa):** ...
   
**Test estadístico:** ...  
**Nivel de significancia alpha:** ...

In [ ]:
# tu código aquí
experiment = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv")
print (experiment.info())
print("\n"+"-"*50+"\n")
print (experiment.head())
print("\n"+"-"*50+"\n")
print (experiment["variante"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB
None

--------------------------------------------------

   id_usuario     variante  convirtio dispositivo       pais  duracion_sesion  \
0  exp_user_0  tratamiento          0      mobile  Argentina           114.41   
1  exp_user_1  tratamiento          0     desktop     Mexico           170.03   
2  exp_user_2      control          1      mobile   Colombia           140.21   
3  exp_user_3  tratamiento       

In [ ]:
from statsmodels.stats.proportion import proportions_ztest
conversiones = experiment.groupby('variante')['convirtio'].sum()
totales = experiment.groupby('variante')['convirtio'].count()
exitos = [conversiones['control'], conversiones['tratamiento']]
observaciones = [totales['control'], totales['tratamiento']]
z_stat, p_value = proportions_ztest(exitos, observaciones)
print(f"Estadístico z: {z_stat}")
print(f"Valor p: {p_value}")

Estadístico z: -0.8132782986429474
Valor p: 0.41605851639119995


In [ ]:
alpha = 0.05
if p_value < alpha:
    print("Rechazamos hipotesis nula: La tasa de conversión es diferente entre el grupo control y el grupo tratamiento")
else:
    print("No rechazamos la hipotesis nula: La tasa de conversión es igual entre el grupo control y el grupo tratamiento")

No rechazamos la hipotesis nula: La tasa de conversión es igual entre el grupo control y el grupo tratamiento


---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión.

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [ ]:
# (Pega aquí tu link)
# link de power bi o tableau
# link de one drive / google drive
https://drive.google.com/drive/folders/1gYYJYgtvt7dDkJuzzaQpICLc_NtLl2Rr